# 01 – Análisis Exploratorio de Datos (EDA)

**TFM:** Sistema Inteligente de Gestión Energética
**Autor:** Álvaro Bermejo Urgel

## Objetivos del notebook

1. Cargar y caracterizar el dataset sintético de las cuatro sedes.
2. Verificar patrones temporales: diario, semanal, anual.
3. Analizar la relación entre variables clave (temperatura, ocupación, consumo).
4. Identificar las anomalías inyectadas (validación del generador).
5. Comparar comportamiento por climatología.

Este EDA establece la base que justificará las decisiones de modelado en el notebook 02.

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["figure.dpi"] = 100

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "synthetic"
print(f"Cargando datos desde: {DATA_DIR}")

## 1. Carga del dataset consolidado

In [ ]:
df = pd.read_parquet(DATA_DIR / "consolidado.parquet")
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.set_index("timestamp").sort_index()

print(f"Filas: {len(df):,}")
print(f"Periodo: {df.index.min()} → {df.index.max()}")
print(f"Sedes: {df['sede'].unique().tolist()}")
df.head()

In [ ]:
# Resumen estadístico por sede
df.groupby("sede")[[
    "temperatura_exterior_c",
    "temperatura_interior_c",
    "humedad_exterior_pct",
    "co2_ppm",
    "ocupacion_rel",
    "consumo_total_kwh",
]].describe().round(2)

In [ ]:
# Verificación de nulos y completitud temporal
print("Valores nulos por columna:")
print(df.isnull().sum())
print(f"\nGaps temporales por sede:")
for sede in df['sede'].unique():
    s = df[df['sede'] == sede].index
    delta = s.to_series().diff().dropna()
    gaps = (delta != pd.Timedelta('1h')).sum()
    print(f"  {sede}: {gaps} gaps")

## 2. Caracterización climática por sede

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 9), sharex=True)
for ax, sede in zip(axes.flat, df['sede'].unique()):
    sub = df[df['sede'] == sede]['temperatura_exterior_c'].resample('D').mean()
    ax.plot(sub.index, sub.values, linewidth=0.8)
    ax.set_title(f"T exterior diaria media – {sede.capitalize()}")
    ax.set_ylabel("°C")
    ax.set_ylim(-5, 40)
plt.tight_layout()
plt.show()

In [ ]:
# Distribución de T exterior por sede (violin plot)
fig, ax = plt.subplots(figsize=(12, 5))
sns.violinplot(data=df, x="sede", y="temperatura_exterior_c", ax=ax, inner="quartile")
ax.set_title("Distribución de temperatura exterior por sede (2 años)")
ax.set_ylabel("Temperatura exterior (°C)")
plt.tight_layout()
plt.show()

## 3. Patrón horario y semanal del consumo

In [ ]:
# Heatmap: hora del día × día de la semana
df_madrid = df[df['sede'] == 'madrid'].copy()
df_madrid['hora'] = df_madrid.index.hour
df_madrid['dia_semana'] = df_madrid.index.dayofweek

pivot = df_madrid.pivot_table(
    values='consumo_total_kwh',
    index='hora',
    columns='dia_semana',
    aggfunc='mean'
)
pivot.columns = ['Lun', 'Mar', 'Mié', 'Jue', 'Vie', 'Sáb', 'Dom']

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(pivot, cmap="YlOrRd", annot=False, cbar_kws={'label': 'kWh'}, ax=ax)
ax.set_title("Consumo medio Madrid: hora × día de la semana")
ax.set_ylabel("Hora del día")
ax.set_xlabel("Día de la semana")
plt.tight_layout()
plt.show()

In [ ]:
# Perfil horario medio por sede
fig, ax = plt.subplots(figsize=(14, 5))
for sede in df['sede'].unique():
    perfil = df[df['sede'] == sede].groupby(df[df['sede'] == sede].index.hour)['consumo_total_kwh'].mean()
    ax.plot(perfil.index, perfil.values, marker='o', label=sede.capitalize(), linewidth=2)
ax.set_xlabel("Hora del día")
ax.set_ylabel("Consumo medio (kWh)")
ax.set_title("Perfil horario medio de consumo por sede")
ax.legend()
ax.set_xticks(range(0, 24))
plt.tight_layout()
plt.show()

## 4. Descomposición del consumo total

In [ ]:
# Stacked area chart: descomposición de consumo en una semana típica
semana = df[(df['sede'] == 'madrid') & (df.index >= '2024-03-04') & (df.index < '2024-03-11')]

fig, ax = plt.subplots(figsize=(15, 6))
ax.stackplot(
    semana.index,
    semana['consumo_base_kwh'],
    semana['consumo_equipos_kwh'],
    semana['consumo_iluminacion_kwh'],
    semana['consumo_hvac_kwh'],
    labels=['Base', 'Equipos', 'Iluminación', 'HVAC'],
    alpha=0.85,
)
ax.set_title("Descomposición del consumo – Madrid, semana del 4-10 marzo 2024")
ax.set_ylabel("kWh")
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

## 5. Relaciones entre variables

In [ ]:
# Matriz de correlación (solo laborables, sin anomalías)
df_clean = df[(~df['es_finde']) & (~df['es_festivo']) & (~df['es_anomalia'])]
df_corr = df_clean[[
    'temperatura_exterior_c',
    'temperatura_interior_c',
    'humedad_exterior_pct',
    'radiacion_solar_rel',
    'ocupacion_rel',
    'co2_ppm',
    'consumo_total_kwh',
    'consumo_hvac_kwh',
    'consumo_iluminacion_kwh',
]].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df_corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax, vmin=-1, vmax=1)
ax.set_title("Matriz de correlación (días laborables, sin anomalías)")
plt.tight_layout()
plt.show()

In [ ]:
# Relación T exterior vs consumo HVAC, por sede
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5), sharey=True)
for ax, sede in zip(axes, df['sede'].unique()):
    sub = df[(df['sede'] == sede) & (df['ocupacion_rel'] > 0.3)].sample(3000, random_state=42)
    ax.scatter(sub['temperatura_exterior_c'], sub['consumo_hvac_kwh'], alpha=0.3, s=8)
    ax.set_title(sede.capitalize())
    ax.set_xlabel("T exterior (°C)")
axes[0].set_ylabel("Consumo HVAC (kWh)")
plt.suptitle("Forma de U: HVAC consume más en frío y en calor extremos", y=1.05)
plt.tight_layout()
plt.show()

## 6. Validación de anomalías inyectadas

In [ ]:
anomalias = df[df['es_anomalia']]
print(f"Total anomalías: {len(anomalias)} ({len(anomalias)/len(df)*100:.2f}% del dataset)")
print("\nDistribución por tipo:")
print(anomalias['tipo_anomalia'].value_counts())
print("\nDistribución por sede:")
print(anomalias['sede'].value_counts())

In [ ]:
# Visualizar una semana con anomalías
fig, ax = plt.subplots(figsize=(16, 5))
muestra = df[df['sede'] == 'madrid'].iloc[:24*30]
ax.plot(muestra.index, muestra['consumo_total_kwh'], linewidth=0.8, label='Consumo')
anom = muestra[muestra['es_anomalia']]
ax.scatter(anom.index, anom['consumo_total_kwh'], color='red', s=30, zorder=5, label='Anomalía')
ax.set_title("Consumo total con anomalías marcadas – Madrid, primer mes")
ax.set_ylabel("kWh")
ax.legend()
plt.tight_layout()
plt.show()

## 7. Estacionalidad anual del consumo HVAC

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
for sede in df['sede'].unique():
    sub = df[df['sede'] == sede]['consumo_hvac_kwh'].resample('W').sum()
    ax.plot(sub.index, sub.values, label=sede.capitalize(), linewidth=1.4)
ax.set_title("Consumo HVAC semanal por sede (2 años)")
ax.set_ylabel("kWh / semana")
ax.legend()
plt.tight_layout()
plt.show()

## 8. KPIs agregados por sede (energía total y patrones)

In [ ]:
kpis = df.groupby('sede').agg(
    consumo_anual_kwh=('consumo_total_kwh', lambda s: s.sum() / 2),  # /2 porque son 2 años
    pct_hvac=('consumo_hvac_kwh', lambda s: s.sum() / df.loc[s.index, 'consumo_total_kwh'].sum() * 100),
    t_ext_media=('temperatura_exterior_c', 'mean'),
    t_int_media=('temperatura_interior_c', 'mean'),
    co2_max=('co2_ppm', 'max'),
).round(2)
kpis

## 9. Conclusiones del EDA

*A rellenar tras ejecutar el notebook:*

1. **Estacionalidad**: confirmada en el consumo total (picos en invierno y verano por HVAC) y en la temperatura.
2. **Patrones horarios**: claros picos en horario laboral (9-18h), valle nocturno y fines de semana.
3. **Diferencias por sede**: …
4. **Correlaciones**: …
5. **Anomalías**: …

Estas conclusiones justifican las decisiones del siguiente notebook:
- Modelos SARIMAX con estacionalidad horaria (s=24) y/o semanal (s=168).
- Variables exógenas críticas: temperatura exterior, ocupación, día de la semana.
- Necesidad de tratar el régimen frío/calor por separado o con interacciones.